# Part 1: HA Dataset Exploration

## Objective
Collect and analyze influenza hemagglutinin (HA) structures to establish baseline dataset for antibody design pipeline.

## Target Structures
- **3LZG**: H1N1 HA structure
- **4O5N**: H3N2 HA structure  
- **5XKU**: H7N9 HA in complex with neutralizing antibody HNIgGA6

## Methods
1. Download PDB structures
2. Analyze structure quality and completeness
3. Extract sequence information
4. Visualize key structural features
5. Document findings for next pipeline steps

In [ ]:
import os
import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from Bio.PDB import PDBParser, PDBList
from Bio.PDB.PDBIO import PDBIO
from Bio.SeqUtils import molecular_weight
import py3Dmol

# Set up visualization
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

print("Dependencies loaded successfully!")

## 1. PDB Structure Download

In [ ]:
# Define target PDB IDs
target_pdbs = {
    '3LZG': 'H1N1 HA structure',
    '4O5N': 'H3N2 HA structure',
    '5XKU': 'H7N9 HA-antibody complex'
}

# Create directories
os.makedirs('../data/raw', exist_ok=True)
os.makedirs('../data/processed', exist_ok=True)

# Download structures
pdbl = PDBList()

for pdb_id, description in target_pdbs.items():
    print(f"Downloading {pdb_id}: {description}")
    
    try:
        pdbl.retrieve_pdb_file(pdb_id, pdir='../data/raw/', file_format='pdb')
        print(f"✅ {pdb_id} downloaded successfully")
    except Exception as e:
        print(f"❌ Error downloading {pdb_id}: {str(e)}")

print("\nDownload phase completed!")

## 2. Structure Analysis

In [ ]:
def analyze_pdb_structure(pdb_file, pdb_id):
    """Analyze PDB structure and extract key information"""
    
    parser = PDBParser(QUIET=True)
    structure = parser.get_structure(pdb_id, pdb_file)
    
    analysis = {
        'pdb_id': pdb_id,
        'models': len(structure),
        'chains': [],
        'residue_count': 0,
        'atom_count': 0,
        'resolution': None,
        'has_antibody': False
    }
    
    for model in structure:
        for chain in model:
            chain_info = {
                'id': chain.id,
                'residues': len([r for r in chain if r.id[0] == ' ']),
                'atoms': len([a for r in chain for a in r if r.id[0] == ' '])
            }
            analysis['chains'].append(chain_info)
            analysis['residue_count'] += chain_info['residues']
            analysis['atom_count'] += chain_info['atoms']
    
    # Check for antibody chains (typically H and L)
    chain_ids = [c['id'] for c in analysis['chains']]
    if 'H' in chain_ids and 'L' in chain_ids:
        analysis['has_antibody'] = True
    
    return analysis

# Analyze all structures
structure_analyses = []

for pdb_id in target_pdbs.keys():
    pdb_file = f"../data/raw/pdb{pdb_id.lower()}.ent"
    
    if os.path.exists(pdb_file):
        analysis = analyze_pdb_structure(pdb_file, pdb_id)
        structure_analyses.append(analysis)
        
        print(f"\n📊 {pdb_id} Analysis:")
        print(f"   Chains: {[c['id'] for c in analysis['chains']]}")
        print(f"   Total residues: {analysis['residue_count']}")
        print(f"   Has antibody: {'Yes' if analysis['has_antibody'] else 'No'}")
    else:
        print(f"❌ {pdb_file} not found")

## 3. Visualization

In [ ]:
def visualize_structure_3d(pdb_file, pdb_id, width=800, height=600):
    """Create 3D visualization of PDB structure"""
    
    # Read PDB content
    with open(pdb_file, 'r') as f:
        pdb_content = f.read()
    
    # Create py3Dmol viewer
    viewer = py3Dmol.view(width=width, height=height)
    viewer.addModel(pdb_content, 'pdb')
    
    # Style settings
    viewer.setStyle({'cartoon': {'color': 'spectrum'}})
    
    # Add labels for chains
    viewer.addLabel(f'{pdb_id}', {'position': {'x': 0, 'y': 0, 'z': 0}, 
                                  'backgroundColor': 'black', 
                                  'fontColor': 'white'})
    
    viewer.zoomTo()
    
    return viewer

# Visualize each structure
for pdb_id in target_pdbs.keys():
    pdb_file = f"../data/raw/pdb{pdb_id.lower()}.ent"
    
    if os.path.exists(pdb_file):
        print(f"\n🔬 Visualizing {pdb_id}:")
        viewer = visualize_structure_3d(pdb_file, pdb_id)
        viewer.show()

## 4. Summary & Next Steps

In [ ]:
# Create summary DataFrame
summary_df = pd.DataFrame([
    {
        'PDB_ID': analysis['pdb_id'],
        'Description': target_pdbs[analysis['pdb_id']],
        'Chains': len(analysis['chains']),
        'Total_Residues': analysis['residue_count'],
        'Has_Antibody': analysis['has_antibody'],
        'Chain_IDs': ', '.join([c['id'] for c in analysis['chains']])
    } for analysis in structure_analyses
])

print("📋 Dataset Summary:")
print(summary_df.to_string(index=False))

# Save summary
summary_df.to_csv('../results/dataset_summary.csv', index=False)

print("\n✅ Part 1 completed successfully!")
print("\n🚀 Next steps:")
print("   - Proceed to Part 2: Structure Preprocessing")
print("   - Focus on 5XKU for antibody-antigen complex analysis")
print("   - Clean structures and prepare for CDR mapping")